In [1]:
!unzip -q aigc_adaptive_detector.zip


In [2]:
%cd aigc_adaptive_detector

/content/aigc_adaptive_detector


In [3]:
!pip install -q -r requirements.txt

In [4]:
import torch

In [5]:
print(torch.__version__, torch.cuda.is_available())

2.11.0+cu128 True


(dummy encoder) — this step only confirms the code runs in this environment.

In [6]:
!python main.py --synthetic --offline --n_calib_per_class 15 --image_size 64

Stage 1 thresholds:      tau_R=-0.2444  tau_F=0.2556
Stage 1+2 thresholds:    tau_R=-0.2401  tau_F=0.2599

Method                 AUC (%)   Avg. cost    Cost %
A_stage1_only              0.0        0.10      8.3%
B_stage1+2               100.0        0.30     25.0%
C_stage1+2+3               0.0        1.20    100.0%
D_adaptive                 0.0        1.20    100.0%


In [9]:
!pip install -q datasets

In [10]:
from datasets import load_dataset
from pathlib import Path

# Slicing keeps the download small -- ~1000 real + ~1000 fake is plenty for the pilot
N_PER_CLASS = 1000

ds = load_dataset("TheKernel01/Tiny-GenImage", split="train")  # streams/downloads parquet shards

root = Path("/content/genimage_subset")
(root / "real").mkdir(parents=True, exist_ok=True)
(root / "fake").mkdir(parents=True, exist_ok=True)

count = {0: 0, 1: 0}
for i, example in enumerate(ds):
    label = example["label"]          # 0 = real, 1 = fake
    if count[label] >= N_PER_CLASS:
        if count[0] >= N_PER_CLASS and count[1] >= N_PER_CLASS:
            break
        continue
    subdir = "real" if label == 0 else "fake"
    img = example["image"].convert("RGB")
    img.save(root / subdir / f"{i:06d}.jpg", quality=95)
    count[label] += 1

print(count)

README.md:   0%|          | 0.00/4.35k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  477MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  475MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

data/validation-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  425MB            

data/validation-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7000 [00:00<?, ? examples/s]

{0: 1000, 1: 1000}


In [11]:
!ls /content/genimage_subset/real | wc -l
!ls /content/genimage_subset/fake | wc -l

!python main.py --data_root /content/genimage_subset --backbone dinov2_vits14 --n_calib_per_class 200

1000
1000
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100% 84.2M/84.2M [00:00<00:00, 281MB/s]
Stage 1 thresholds:      tau_R=-0.1969  tau_F=0.3031
Stage 1+2 thresholds:    tau_R=-0.1786  tau_F=0.3214

Method                

In [13]:
!python main.py --data_root /content/genimage_subset --backbone dinov2_vits14 --n_calib_per_class 500

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Stage 1 thresholds:      tau_R=-0.1958  tau_F=0.3042
Stage 1+2 thresholds:    tau_R=-0.1801  tau_F=0.3199

Method                 AUC (%)   Avg. cost    Cost %
A_stage1_only             55.4        2.00      8.3%
B_stage1+2                59.4        6.00     25.0%
C_stage1+2+3              65.0       24.00    100.0%
D_adaptive                65.1       23.78     99.1%

In [14]:
!python main.py --data_root /content/genimage_subset --backbone dinov2_vits14 --n_calib_per_class 1000

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Stage 1 thresholds:      tau_R=-0.1958  tau_F=0.3042
Stage 1+2 thresholds:    tau_R=-0.1801  tau_F=0.3199

Method                 AUC (%)   Avg. cost    Cost %
A_stage1_only             55.4        2.00      8.3%
B_stage1+2                59.4        6.00     25.0%
C_stage1+2+3              65.0       24.00    100.0%
D_adaptive                65.1       23.78     99.1%